# Exploration — Enveloppe thermique
Analyse des niveaux d isolation, de l etancheite a l air et leur impact sur la consommation — ResStock 2025

In [ ]:
import re
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT           = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES        = ROOT / 'reports' / 'figures'

df = pd.read_parquet(DATA_PROCESSED / 'metadata_clean.parquet')
CONSO_COL     = 'out.electricity.total.energy_consumption..kwh'
INTENSITY_COL = 'out.electricity.total.energy_consumption_intensity..kwh_per_ft2'
df[CONSO_COL]     = pd.to_numeric(df[CONSO_COL], errors='coerce')
df[INTENSITY_COL] = pd.to_numeric(df[INTENSITY_COL], errors='coerce')
print('Shape:', df.shape)

## 1. Isolation par composant — R-values ordonnees

Pour chaque composant (plafond, toiture, murs), les valeurs R sont extraites et ordonnees numeriquement.
L axe droit montre l **intensite de consommation (kWh/ft2)** — normalise par la surface pour eviter le biais taille.

In [ ]:
def extract_r_value(val):
    if pd.isna(val) or str(val).strip() == 'None':
        return None
    s = str(val)
    if 'Uninsulated' in s:
        return 0
    m = re.search(r'R-(\d+)', s)
    return int(m.group(1)) if m else None

INSUL_COLS = {
    'Plafond (insulation_ceiling)': 'in.insulation_ceiling',
    'Toiture (insulation_roof)':    'in.insulation_roof',
    'Murs (insulation_wall)':       'in.insulation_wall',
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (titre, col) in zip(axes, INSUL_COLS.items()):
    tmp = df[[col, CONSO_COL]].copy()
    tmp['r_val'] = tmp[col].apply(extract_r_value)
    tmp = tmp.dropna(subset=['r_val', CONSO_COL])

    grp = tmp.groupby('r_val').agg(
        count=(col, 'count'),
        conso_moy=(CONSO_COL, 'mean')
    ).sort_index()

    labels = ['Non isole' if r == 0 else f'R-{int(r)}' for r in grp.index]
    x = list(range(len(labels)))

    # Barres = nb batiments (axe gauche)
    ax.bar(x, grp['count'], color='#5b9bd5', alpha=0.75, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_ylabel('Nb batiments', color='#5b9bd5', fontsize=9)
    ax.tick_params(axis='y', labelcolor='#5b9bd5')
    ax.set_title(titre, fontweight='bold', fontsize=10)
    ax.spines['top'].set_visible(False)

    # Ligne = consommation moy (axe droit)
    ax2 = ax.twinx()
    ax2.plot(x, grp['conso_moy'], color='#e74c3c', marker='o',
             linewidth=2.5, markersize=7)
    ax2.set_ylabel('Consommation moy. (kWh)', color='#e74c3c', fontsize=9)
    ax2.tick_params(axis='y', labelcolor='#e74c3c')
    ax2.spines['top'].set_visible(False)

fig.suptitle(
    "Enveloppe thermique — Nb batiments (barres) et consommation moyenne kWh (ligne) par niveau d'isolation",
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.savefig(FIGURES / 'enveloppe_r_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('Sauvegarde : enveloppe_r_values.png')

## 2. Etancheite a l air — ACH50

 est la seule colonne d enveloppe quantitative continue.
Un ACH50 eleve = batiment tres permeable = pertes thermiques importantes.